In [1]:
#pip install scikit-learn pandas numpy matplotlib qiskit
#pip qiskit-machine-learning

In [2]:
from qiskit import QuantumCircuit

from qiskit.circuit import Parameter
from qiskit.circuit import ParameterVector

from qiskit.primitives import StatevectorSampler

from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.state_fidelities import ComputeUncompute

# Quantum Kernel Implementation

In [3]:
def build_feature_map(n_qubits = 8):
    x = Parameter("x")
    qc = QuantumCircuit(n_qubits, name = "FeatureMap")
    
    for q in range(n_qubits):
        qc.rx((q * x) / 2, q)
        
    return qc


def build_HEA(n_qubits = 8, seed = 1):
    depth = 5
    rng = np.random.default_rng(seed = seed)
    theta = rng.uniform(0, 2 * np.pi, size = (depth, n_qubits, 2))
    
    qc = QuantumCircuit(n_qubits, name = "HEA")
    
    idx = 0
    for i in range(depth):
        
        for q in range(n_qubits):
            qc.rx(theta[i, q, 0], q)
            qc.rz(theta[i, q, 1], q)
            idx += 2
            
        for q in range(n_qubits - 1):
            qc.cx(q, q+1)
            
    return qc

In [4]:
def build_kernel_feature_map(n_qubits = 8):
    qc = QuantumCircuit(n_qubits, name = "KernelMap")
    
    qc.compose(build_HEA(n_qubits), inplace = True)
    qc.compose(build_feature_map(n_qubits), inplace = True)
    
    return qc

def build_quantum_kernel(n_qubits = 8):
    
    feature_map = build_kernel_feature_map(n_qubits)
    
    sampler = StatevectorSampler()
    fidelity = ComputeUncompute(sampler=sampler)
    
    return FidelityQuantumKernel(fidelity = fidelity, feature_map = feature_map)

### Usage

In [7]:
kernel = build_quantum_kernel(8)